# Preexperimento (2/3): *Screening* de columnas por fuente

Documenta los resultados del *screening* ligero de columnas por fuente/familia. **No** elige el mejor modelo ni exporta figuras: identifica columnas útiles, redundantes e inestables por combinación de símbolo, método de etiquetado, subconjunto de fuente y familia de modelado.

La corrida esperada evalúa `core_only` (OHLCV/base técnica) y `core_only + fuente` para `derivatives_futures`, `onchain`, `microstructure`, `sentiment` y `attention`, cruzado con `fixed_horizon`, `triple_barrier`, `trend_scanning`, `xgboost` y `lstm`.

**Relación con el resto del preexperimento:** este notebook y `01_evaluacion_fracdiff.ipynb` son evidencias paralelas; `03_presentacion_screening_variables_y_ffd_por_fuente.ipynb` actúa como síntesis visual para presentación.

## Qué responde este notebook

1. Cobertura y criterios de la corrida congelada (`run_id`, celdas completadas, selectores y backends observados).
2. Resumen por fuente: columnas evaluadas, seleccionadas y recomendadas por celda experimental.
3. Detalle por columna: frecuencia de selección, redundancia e importancia media.
4. Agregado por fuente y familia para priorizar señal estable antes del diseño confirmatorio.

## Alcance

- Entrada canónica: `reports/validation/source_column_screening/latest_manifest.json` -> CSV del directorio de corrida referenciado en `paths`.
- **Solo lectura:** no hay `savefig` ni export tabular desde este cuaderno; las figuras de memoria viven en `03_presentacion_screening_variables_y_ffd_por_fuente.ipynb` bajo `reports/figures/presentacion/`.
- Los recuentos de la tabla ejecutiva se leen del manifest y de los CSV en cada ejecución; no fijar cifras en prosa.

## Productos consumidos (lectura)

Tras resolver `latest_manifest.json`, el notebook carga (rutas relativas al repo):

- `manifest.json`
- `cell_summary.csv`
- `column_summary.csv`
- `fold_importance.csv`
- `source_summary.csv`

Todos bajo `reports/validation/source_column_screening/<run_id>/`.


In [1]:
from pathlib import Path
import json

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)


def _resolve_repo_root() -> Path:
    """Sube desde cwd (p. ej. .../preexperimento) hasta la raíz del repo."""
    cwd = Path.cwd().resolve()
    for p in (cwd, *cwd.parents):
        if (p / "pytest.ini").is_file() and (p / "src").is_dir():
            return p
    for p in (cwd, *cwd.parents):
        d = p / "reports" / "validation" / "source_column_screening"
        if d.is_dir():
            return p
    return cwd


REPO_ROOT = _resolve_repo_root()

REPORT_ROOT = REPO_ROOT / "reports" / "validation" / "source_column_screening"
LATEST_MANIFEST = REPORT_ROOT / "latest_manifest.json"

if not LATEST_MANIFEST.exists():
    raise FileNotFoundError(f"No existe el manifiesto latest: {LATEST_MANIFEST}")

manifest = json.loads(LATEST_MANIFEST.read_text(encoding="utf-8"))
run_dir = REPO_ROOT / manifest["paths"]["manifest_json"]
run_dir = run_dir.parent

manifest

{'completed_cells': 180,
 'config': {'allow_lstm_fallback': False,
  'families': ['xgboost', 'lstm'],
  'features_root': 'data/02_intermediate',
  'lstm_batch_size': 64,
  'lstm_learning_rate': 0.001,
  'lstm_lookback_bars': 42,
  'lstm_max_epochs': 3,
  'lstm_permutation_repeats': 1,
  'max_folds': 3,
  'methods': ['fixed_horizon', 'triple_barrier', 'trend_scanning'],
  'min_importance': 0.0,
  'processed_root': 'data/03_processed',
  'redundancy_corr_threshold': 0.95,
  'source_subset_ids': ['core_only',
   'derivatives_futures',
   'onchain',
   'microstructure',
   'sentiment',
   'attention'],
  'step_bars': 500,
  'symbols': ['BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'XRPUSDT', 'SOLUSDT'],
  'test_bars': 500,
  'top_k': 25,
  'train_bars': 2500,
  'xgb_n_estimators': 35},
 'failed_or_skipped_cells': 0,
 'failures': [],
 'generated_at_utc': '2026-04-26T14:29:22.545489+00:00',
 'mode': 'source_column_screening',
 'n_column_summary_rows': 6378,
 'n_failures': 0,
 'n_fold_rows': 18690,
 'n_so

In [2]:
fold_importance = pd.read_csv(REPO_ROOT / manifest["paths"]["fold_importance_csv"])
column_summary = pd.read_csv(REPO_ROOT / manifest["paths"]["column_summary_csv"])
source_summary = pd.read_csv(REPO_ROOT / manifest["paths"]["source_summary_csv"])
cell_summary = pd.read_csv(REPO_ROOT / manifest["paths"]["cell_summary_csv"])

ETIQUETAS_COLUMNAS_ES: dict[str, str] = {
    "symbol": "simbolo",
    "method": "metodo_etiquetado",
    "source_subset_id": "subconjunto_fuente",
    "family": "familia",
    "n_columns_screened": "columnas_evaluadas",
    "n_columns_selected_any_fold": "columnas_seleccionadas_en_algun_fold",
    "n_columns_recommended": "columnas_recomendadas",
    "mean_selection_frequency": "frecuencia_media_seleccion",
    "top_recommended_columns": "top_columnas_recomendadas",
    "column": "columna",
    "column_source": "origen_columna",
    "n_folds": "folds_totales",
    "selected_folds": "folds_seleccionados",
    "selection_frequency": "frecuencia_seleccion",
    "redundant_folds": "folds_redundantes",
    "redundancy_frequency": "frecuencia_redundancia",
    "mean_importance": "importancia_media",
    "max_importance": "importancia_maxima",
    "median_rank": "ranking_mediano",
    "most_common_redundant_with": "redundante_con_frecuente",
    "recommended": "recomendada",
    "n_columns": "columnas_unicas",
    "n_recommended": "recomendadas",
    "mean_redundancy": "redundancia_media",
}


def etiquetar_columnas(df: pd.DataFrame) -> pd.DataFrame:
    """Renombra columnas conocidas al español para tablas de presentación."""
    rename = {k: v for k, v in ETIQUETAS_COLUMNAS_ES.items() if k in df.columns}
    return df.rename(columns=rename)


config = manifest["config"]
recommended_count = int(column_summary["recommended"].astype(bool).sum())
redundant_count = int((column_summary["redundancy_frequency"] > 0).sum())
selector_families = ", ".join(sorted(fold_importance["selector_family"].dropna().unique()))
backend_counts = (
    fold_importance["backend_name"].dropna().value_counts().to_dict()
    if "backend_name" in fold_importance.columns
    else {}
)
backend_evidence = ", ".join(
    f"{backend}: {count}" for backend, count in backend_counts.items()
) or "no disponible"

screening_evidence_summary = pd.DataFrame(
    [
        {
            "bloque": "Identidad",
            "métrica": "Corrida analizada",
            "valor": manifest["run_id"],
            "evidencia": "latest_manifest.json",
        },
        {
            "bloque": "Cobertura",
            "métrica": "Celdas planificadas",
            "valor": manifest.get("planned_cells"),
            "evidencia": "manifest.json",
        },
        {
            "bloque": "Cobertura",
            "métrica": "Celdas completadas",
            "valor": manifest.get("completed_cells"),
            "evidencia": "manifest.json",
        },
        {
            "bloque": "Cobertura",
            "métrica": "Celdas fallidas u omitidas",
            "valor": manifest.get("failed_or_skipped_cells"),
            "evidencia": "manifest.json",
        },
        {
            "bloque": "Alcance",
            "métrica": "Símbolos",
            "valor": len(config["symbols"]),
            "evidencia": ", ".join(config["symbols"]),
        },
        {
            "bloque": "Alcance",
            "métrica": "Métodos de etiquetado",
            "valor": len(config["methods"]),
            "evidencia": ", ".join(config["methods"]),
        },
        {
            "bloque": "Alcance",
            "métrica": "Subconjuntos de fuentes",
            "valor": len(config["source_subset_ids"]),
            "evidencia": ", ".join(config["source_subset_ids"]),
        },
        {
            "bloque": "Alcance",
            "métrica": "Familias de criterio",
            "valor": len(config["families"]),
            "evidencia": ", ".join(config["families"]),
        },
        {
            "bloque": "Artefactos",
            "métrica": "Filas de detalle por fold",
            "valor": len(fold_importance),
            "evidencia": "fold_importance.csv",
        },
        {
            "bloque": "Artefactos",
            "métrica": "Filas agregadas por columna",
            "valor": len(column_summary),
            "evidencia": "column_summary.csv",
        },
        {
            "bloque": "Artefactos",
            "métrica": "Filas de resumen por fuente",
            "valor": len(source_summary),
            "evidencia": "source_summary.csv",
        },
        {
            "bloque": "Resultado preliminar",
            "métrica": "Columnas marcadas como recomendadas",
            "valor": recommended_count,
            "evidencia": "recomendada == True en column_summary.csv",
        },
        {
            "bloque": "Resultado preliminar",
            "métrica": "Columnas con redundancia detectada",
            "valor": redundant_count,
            "evidencia": "frecuencia_redundancia > 0 en column_summary.csv",
        },
        {
            "bloque": "Criterio",
            "métrica": "Selectores observados",
            "valor": selector_families,
            "evidencia": "selector_family en fold_importance.csv",
        },
        {
            "bloque": "Criterio",
            "métrica": "Backends de entrenamiento observados",
            "valor": backend_evidence,
            "evidencia": "backend_name en fold_importance.csv",
        },
        {
            "bloque": "Criterio",
            "métrica": "Top-k por fold",
            "valor": config["top_k"],
            "evidencia": "config.top_k",
        },
        {
            "bloque": "Criterio",
            "métrica": "Umbral de redundancia por correlación",
            "valor": config["redundancy_corr_threshold"],
            "evidencia": "config.redundancy_corr_threshold",
        },
        {
            "bloque": "Criterio",
            "métrica": "Folds máximos por celda",
            "valor": config["max_folds"],
            "evidencia": "config.max_folds",
        },
    ]
)

screening_evidence_summary

,bloque,métrica,valor,evidencia
0,Identidad,Corrida analizada,source_screening_lstm_real_20260426T1430,latest_manifest.json
1,Cobertura,Celdas planificadas,180,manifest.json
2,Cobertura,Celdas completadas,180,manifest.json
3,Cobertura,Celdas fallidas u omitidas,0,manifest.json
4,Alcance,Símbolos,5,"BTCUSDT, ETHUSDT, BNBUSDT, XRPUSDT, SOLUSDT"
5,Alcance,Métodos de etiquetado,3,"fixed_horizon, triple_barrier, trend_scanning"
6,Alcance,Subconjuntos de fuentes,6,"core_only, derivatives_futures, onchain, microstructure, sentiment, attention"
7,Alcance,Familias de criterio,2,"xgboost, lstm"
8,Artefactos,Filas de detalle por fold,18690,fold_importance.csv
9,Artefactos,Filas agregadas por columna,6378,column_summary.csv


## Lectura rápida

La tabla de resumen por fuente permite comprobar si una fuente genera columnas recomendadas de forma estable. La tabla agregada por columna baja al detalle, con frecuencia de selección, redundancia y ranking medio.

In [3]:
etiquetar_columnas(
    source_summary.sort_values(
        ["symbol", "method", "source_subset_id", "family"]
    )
).head(30)

,simbolo,metodo_etiquetado,subconjunto_fuente,familia,columnas_evaluadas,columnas_seleccionadas_en_algun_fold,columnas_recomendadas,frecuencia_media_seleccion,top_columnas_recomendadas
0,BNBUSDT,fixed_horizon,attention,lstm,40,30,3,0.633333,"[""obv"", ""reddit_post_count"", ""pvt""]"
1,BNBUSDT,fixed_horizon,attention,xgboost,40,32,24,0.625000,"[""obv"", ""atr_14"", ""num_trades"", ""rolling_std_20"", ""macd_hist"", ""macd_line"", ""slope_sma_20"", ""mfi_14"", ""stoch_d_14"", ..."
2,BNBUSDT,fixed_horizon,core_only,lstm,29,20,4,0.643678,"[""taker_buy_asset_volume"", ""obv"", ""num_trades"", ""pvt""]"
3,BNBUSDT,fixed_horizon,core_only,xgboost,29,22,18,0.643678,"[""obv"", ""atr_14"", ""num_trades"", ""rolling_std_20"", ""macd_signal"", ""macd_hist"", ""slope_sma_20"", ""stoch_d_14"", ""mfi_14""..."
4,BNBUSDT,fixed_horizon,derivatives_futures,lstm,29,20,4,0.643678,"[""taker_buy_asset_volume"", ""obv"", ""num_trades"", ""pvt""]"
5,BNBUSDT,fixed_horizon,derivatives_futures,xgboost,29,22,18,0.643678,"[""obv"", ""atr_14"", ""num_trades"", ""rolling_std_20"", ""macd_signal"", ""macd_hist"", ""slope_sma_20"", ""stoch_d_14"", ""mfi_14""..."
6,BNBUSDT,fixed_horizon,microstructure,lstm,42,30,6,0.595238,"[""obv"", ""ms_total_volume"", ""ms_cvd"", ""pvt"", ""volume"", ""num_trades""]"
7,BNBUSDT,fixed_horizon,microstructure,xgboost,42,31,25,0.595238,"[""pvt"", ""atr_14"", ""ms_trade_count"", ""macd_signal"", ""rolling_std_20"", ""slope_sma_20"", ""ms_roll_spread"", ""macd_hist"", ..."
8,BNBUSDT,fixed_horizon,onchain,lstm,29,20,4,0.643678,"[""taker_buy_asset_volume"", ""obv"", ""num_trades"", ""pvt""]"
9,BNBUSDT,fixed_horizon,onchain,xgboost,29,22,18,0.643678,"[""obv"", ""atr_14"", ""num_trades"", ""rolling_std_20"", ""macd_signal"", ""macd_hist"", ""slope_sma_20"", ""stoch_d_14"", ""mfi_14""..."


In [4]:
recommended = column_summary[column_summary["recommended"] == True].copy()
etiquetar_columnas(
    recommended.sort_values(
        ["symbol", "method", "source_subset_id", "family", "selection_frequency", "mean_importance"],
        ascending=[True, True, True, True, False, False],
    )
).head(50)

,simbolo,metodo_etiquetado,subconjunto_fuente,familia,columna,origen_columna,folds_totales,folds_seleccionados,frecuencia_seleccion,folds_redundantes,frecuencia_redundancia,importancia_media,importancia_maxima,ranking_mediano,redundante_con_frecuente,recomendada
0,BNBUSDT,fixed_horizon,attention,lstm,obv,technical_or_derived,3,3,1.000000,0,0.000000,2.686381e-03,4.366994e-03,3.0,pvt,True
1,BNBUSDT,fixed_horizon,attention,lstm,reddit_post_count,attention,3,3,1.000000,0,0.000000,5.960000e-08,1.788000e-07,22.0,pvt,True
21,BNBUSDT,fixed_horizon,attention,lstm,pvt,technical_or_derived,3,2,0.666667,1,0.333333,3.051738e-03,6.503582e-03,2.0,obv,True
40,BNBUSDT,fixed_horizon,attention,xgboost,obv,technical_or_derived,3,3,1.000000,0,0.000000,5.852230e-02,8.547925e-02,2.0,low,True
41,BNBUSDT,fixed_horizon,attention,xgboost,atr_14,technical,3,3,1.000000,0,0.000000,3.564935e-02,4.995024e-02,17.0,rolling_std_20,True
42,BNBUSDT,fixed_horizon,attention,xgboost,num_trades,ohlcv,3,3,1.000000,0,0.000000,3.524398e-02,4.607308e-02,14.0,rolling_std_20,True
43,BNBUSDT,fixed_horizon,attention,xgboost,rolling_std_20,technical_or_derived,3,3,1.000000,0,0.000000,3.093395e-02,3.544450e-02,15.0,atr_14,True
44,BNBUSDT,fixed_horizon,attention,xgboost,macd_hist,technical,3,3,1.000000,0,0.000000,2.941820e-02,3.951209e-02,22.0,stoch_d_14,True
45,BNBUSDT,fixed_horizon,attention,xgboost,macd_line,technical,3,3,1.000000,0,0.000000,2.752242e-02,3.421579e-02,21.0,macd_signal,True
46,BNBUSDT,fixed_horizon,attention,xgboost,slope_sma_20,technical_or_derived,3,3,1.000000,0,0.000000,2.679714e-02,3.326159e-02,17.0,macd_line,True


## Columnas redundantes

Una columna puede tener importancia alta y aún así ser redundante si queda muy correlacionada con otra columna mejor posicionada dentro del mismo fold. Esta tabla ayuda a detectar variables que conviene no duplicar en la experimentación confirmatoria.

In [5]:
redundant = column_summary[column_summary["redundancy_frequency"] > 0].copy()
etiquetar_columnas(
    redundant.sort_values(
        ["redundancy_frequency", "mean_importance"],
        ascending=[False, False],
    )
).head(50)

,simbolo,metodo_etiquetado,subconjunto_fuente,familia,columna,origen_columna,folds_totales,folds_seleccionados,frecuencia_seleccion,folds_redundantes,frecuencia_redundancia,importancia_media,importancia_maxima,ranking_mediano,redundante_con_frecuente,recomendada
798,BNBUSDT,trend_scanning,sentiment,xgboost,high,ohlcv,3,0,0.0,3,1.0,0.058854,0.078493,4.0,close,False
131,BNBUSDT,fixed_horizon,core_only,xgboost,bbands_upper_20,technical_or_derived,3,0,0.0,3,1.0,0.054900,0.059138,3.0,open,False
189,BNBUSDT,fixed_horizon,derivatives_futures,xgboost,bbands_upper_20,technical_or_derived,3,0,0.0,3,1.0,0.054900,0.059138,3.0,open,False
331,BNBUSDT,fixed_horizon,onchain,xgboost,bbands_upper_20,technical_or_derived,3,0,0.0,3,1.0,0.054900,0.059138,3.0,open,False
3286,ETHUSDT,trend_scanning,sentiment,xgboost,sma_50,technical_or_derived,3,0,0.0,3,1.0,0.053153,0.058583,3.0,open,False
2584,ETHUSDT,fixed_horizon,core_only,xgboost,ema_26,technical_or_derived,3,0,0.0,3,1.0,0.053072,0.059698,2.0,sma_50,False
2642,ETHUSDT,fixed_horizon,derivatives_futures,xgboost,ema_26,technical_or_derived,3,0,0.0,3,1.0,0.053072,0.059698,2.0,sma_50,False
534,BNBUSDT,trend_scanning,core_only,xgboost,high,ohlcv,3,0,0.0,3,1.0,0.051863,0.068529,9.0,close,False
592,BNBUSDT,trend_scanning,derivatives_futures,xgboost,high,ohlcv,3,0,0.0,3,1.0,0.051863,0.068529,9.0,close,False
734,BNBUSDT,trend_scanning,onchain,xgboost,high,ohlcv,3,0,0.0,3,1.0,0.051863,0.068529,9.0,close,False


## Comparativa por fuente y familia

Este agregado cuenta cuántas columnas recomendadas aparecen por fuente y por familia de modelado. Sirve para decidir qué fuentes tienen suficiente señal estable para pasar a la fase de experimentación.

In [6]:
summary_by_source = (
    column_summary.assign(recommended=column_summary["recommended"].astype(bool))
    .groupby(["method", "source_subset_id", "family", "column_source"], dropna=False)
    .agg(
        n_columns=("column", "nunique"),
        n_recommended=("recommended", "sum"),
        mean_selection_frequency=("selection_frequency", "mean"),
        mean_importance=("mean_importance", "mean"),
        mean_redundancy=("redundancy_frequency", "mean"),
    )
    .reset_index()
    .sort_values(["method", "source_subset_id", "family", "n_recommended"], ascending=[True, True, True, False])
)
etiquetar_columnas(summary_by_source).head(80)

,metodo_etiquetado,subconjunto_fuente,familia,origen_columna,columnas_unicas,recomendadas,frecuencia_media_seleccion,importancia_media,redundancia_media
3,fixed_horizon,attention,lstm,technical_or_derived,17,24,0.654902,0.000963,0.301961
0,fixed_horizon,attention,lstm,attention,27,13,0.715152,0.000630,0.181818
1,fixed_horizon,attention,lstm,ohlcv,7,8,0.304762,0.001607,0.638095
2,fixed_horizon,attention,lstm,technical,5,7,0.840000,0.000015,0.160000
7,fixed_horizon,attention,xgboost,technical_or_derived,17,49,0.639216,0.027177,0.337255
...,...,...,...,...,...,...,...,...,...
73,trend_scanning,microstructure,xgboost,ohlcv,7,7,0.285714,0.028726,0.685714
77,trend_scanning,onchain,lstm,onchain,12,31,0.817460,0.001085,0.182540
79,trend_scanning,onchain,lstm,technical_or_derived,17,19,0.631373,0.001371,0.305882
76,trend_scanning,onchain,lstm,ohlcv,7,9,0.285714,0.000975,0.638095


## Vista filtrada (plantilla)

Edita al inicio de la celda de código las variables `symbol`, `method`, `source_subset_id` y `family` para inspeccionar una combinación concreta. Por defecto se muestra la primera combinación disponible en `column_summary.csv`.

In [7]:
symbol = column_summary["symbol"].drop_duplicates().iloc[0]
method = column_summary["method"].drop_duplicates().iloc[0]
source_subset_id = column_summary["source_subset_id"].drop_duplicates().iloc[0]
family = column_summary["family"].drop_duplicates().iloc[0]

view = column_summary.query(
    "symbol == @symbol and method == @method and source_subset_id == @source_subset_id and family == @family"
).sort_values(["recommended", "selection_frequency", "mean_importance"], ascending=[False, False, False])

etiquetar_columnas(view).head(100)

,simbolo,metodo_etiquetado,subconjunto_fuente,familia,columna,origen_columna,folds_totales,folds_seleccionados,frecuencia_seleccion,folds_redundantes,frecuencia_redundancia,importancia_media,importancia_maxima,ranking_mediano,redundante_con_frecuente,recomendada
0,BNBUSDT,fixed_horizon,attention,lstm,obv,technical_or_derived,3,3,1.000000,0,0.000000,2.686381e-03,4.366994e-03,3.0,pvt,True
1,BNBUSDT,fixed_horizon,attention,lstm,reddit_post_count,attention,3,3,1.000000,0,0.000000,5.960000e-08,1.788000e-07,22.0,pvt,True
21,BNBUSDT,fixed_horizon,attention,lstm,pvt,technical_or_derived,3,2,0.666667,1,0.333333,3.051738e-03,6.503582e-03,2.0,obv,True
2,BNBUSDT,fixed_horizon,attention,lstm,atr_14,technical,3,3,1.000000,0,0.000000,0.000000e+00,0.000000e+00,3.0,volume,False
3,BNBUSDT,fixed_horizon,attention,lstm,bbands_lower_20,technical_or_derived,3,3,1.000000,0,0.000000,0.000000e+00,0.000000e+00,4.0,atr_14,False
4,BNBUSDT,fixed_horizon,attention,lstm,bbands_pctb_20,technical_or_derived,3,3,1.000000,0,0.000000,0.000000e+00,0.000000e+00,5.0,volume,False
5,BNBUSDT,fixed_horizon,attention,lstm,gt_bnb,attention,3,3,1.000000,0,0.000000,0.000000e+00,0.000000e+00,10.0,bbands_lower_20,False
6,BNBUSDT,fixed_horizon,attention,lstm,gt_bnb_lag_1,attention,3,3,1.000000,0,0.000000,0.000000e+00,0.000000e+00,11.0,gt_bnb,False
7,BNBUSDT,fixed_horizon,attention,lstm,gt_bnb_price,attention,3,3,1.000000,0,0.000000,0.000000e+00,0.000000e+00,12.0,bbands_lower_20,False
8,BNBUSDT,fixed_horizon,attention,lstm,gt_bnb_price_lag_1,attention,3,3,1.000000,0,0.000000,0.000000e+00,0.000000e+00,13.0,atr_14,False
